In [83]:
from langgraph.graph import START, END, StateGraph
from langchain_openai import ChatOpenAI

from typing import Annotated, List, Optional, Literal
from pydantic import BaseModel, Field

from langgraph.checkpoint.memory import InMemorySaver

In [84]:
"""
- 2 line doc 
"""

'\n- 2 line doc \n'

In [85]:
smallModel = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.0, max_tokens=1000)
bigModel = ChatOpenAI(model_name="gpt-4", temperature=0.0, max_tokens=1000)

In [86]:
class Content(BaseModel):
    topic: str = Field(..., description="The topic of the content.")
    content: str = Field(default="", description="The content generated.")
    review: Optional[Literal["good", "bad"]] = Field(
        default=None, description="Review of the content, either 'good' or 'bad'."
    )
    attempts: int = Field(default=0, description="Number of generation attempts so far.")


In [87]:
MAX_ATTEMPTS = 3


def generate_content(state: Content) -> Content:
    topic = state.topic
    prompt = f"Generate a piece of content about {topic}."
    response = smallModel.invoke(prompt)
    state.content = response.content
    state.attempts += 1
    return state


def review_content(state: Content) -> Content:
    content = state.content
    prompt = f"Review the following content and provide feedback: {content}"
    response = bigModel.invoke(prompt)
    review = "good" if "good" in response.content.lower() else "bad"
    state.review = review
    return state

def navigation(state: Content) -> str:
    if state.review == "good" or state.attempts >= MAX_ATTEMPTS:
        return 'done'
    else:
        return 'retry'


In [88]:
state_graph = StateGraph(Content)

In [ ]:
state_graph.add_node('generate_content', generate_content)
state_graph.add_node('review_content', review_content)

state_graph.add_edge(START, 'generate_content')
state_graph.add_edge('generate_content', 'review_content')
state_graph.add_conditional_edges('review_content', navigation, {
    'done': END,
    'retry': 'generate_content'
})

checkpoint_saver = InMemorySaver()
workflow = state_graph.compile(checkpointer=checkpoint_saver)



In [94]:
workflow
configuration = {"configurable": {"thread_id": "1"}}


In [95]:
response1 = workflow.invoke({'topic': 'Artificial Intelligence'}, config=configuration)
response1

{'topic': 'Artificial Intelligence',
 'content': 'Artificial Intelligence, or AI, is a rapidly advancing technology that is revolutionizing industries across the globe. From self-driving cars to virtual assistants, AI is changing the way we live and work.\n\nOne of the key benefits of AI is its ability to analyze vast amounts of data quickly and accurately. This has huge implications for fields such as healthcare, where AI can help doctors diagnose diseases more effectively and develop personalized treatment plans for patients.\n\nAI is also being used in the financial sector to detect fraud and make investment decisions, as well as in the transportation industry to optimize routes and reduce traffic congestion.\n\nHowever, as AI becomes more sophisticated, there are also concerns about its potential impact on jobs and privacy. Some fear that AI will lead to widespread job loss as machines take over tasks that were previously done by humans. There are also concerns about the ethical im

In [112]:
history = workflow.get_state_history(configuration)

for state in history:
    print(state)

StateSnapshot(values={'topic': 'Machine Learning', 'content': 'Machine learning is a branch of artificial intelligence that focuses on developing algorithms and models that allow computers to learn from and make predictions or decisions based on data. This technology has revolutionized various industries, from healthcare to finance to marketing, by enabling organizations to analyze large amounts of data and extract valuable insights.\n\nOne of the key benefits of machine learning is its ability to automate and optimize processes that would be time-consuming or impossible for humans to perform. For example, in healthcare, machine learning algorithms can analyze medical images to detect diseases or predict patient outcomes. In finance, machine learning models can analyze market trends and make investment decisions in real-time.\n\nMachine learning also plays a crucial role in improving customer experiences and personalizing services. By analyzing customer data, organizations can create t

In [107]:
# updating state on node 1
old_config = {"configurable": {"thread_id": "1", "checkpoint_ns":"", "checkpoint_id":"1f1a6bfd-8b02-6e8a-8000-2a9bf3728656"}}

workflow.update_state(old_config, {"topic": "Machine Learning"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1a6c05-a5ef-601a-8001-03053db69c3e'}}

In [111]:
# re-run workflow
updated_config = {"configurable": {"thread_id": "1", "checkpoint_ns":"", "checkpoint_id":"1f1a6c05-a5ef-601a-8001-03053db69c3e"}}
response2 = workflow.invoke(None, config=updated_config)
